In [ ]:
!nvidia-smi

Wed Sep 23 08:40:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip-install -U "transformers>=4.45.0" accelerate

/bin/bash: line 1: pip-install: command not found


In [ ]:
!pip	-q	install	-U	"transformers>=4.45.0"	accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.5 MB/s eta 0:00:00


In [ ]:
import	torch,	json,	re
from	transformers	import	pipeline
MODEL_ID	=	"Qwen/Qwen2.5-1.5B-Instruct"
#	GPU	runtime
#	MODEL_ID	=	"Qwen/Qwen2.5-0.5B-Instruct"			#	<--	switch	to	this	line	if	you	are	on	CPU
llm	=	pipeline(
"text-generation",
model=MODEL_ID,
torch_dtype=torch.bfloat16	if	torch.cuda.is_available()	else	torch.float32,
device_map="auto",
)
print("Running	on:",	"GPU"	if	torch.cuda.is_available()	else	"CPU")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Running	on: GPU


In [ ]:
def ask(system_prompt, user_message, sample=False, temperature=0.7, max_new_tokens=120):
    """Send one system prompt + one user message to the model, return the text reply."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",
         "content": user_message},
    ]
    kwargs = {"max_new_tokens": max_new_tokens, "return_full_text": False}
    if sample:
        kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        kwargs.update(do_sample=False)
    # greedy = same input, same output
    out = llm(messages, **kwargs)[0]["generated_text"]
    if isinstance(out, list):
        # some versions return the full chat
        out = out[-1]["content"]
    return out.strip()


# smoke test
print(ask("You are a terse assistant.", "Say OK and nothing else."))

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


OK


In [ ]:
LABELS = ["billing", "technical", "account_access", "other"]

TEST_SET = [
  ("I was charged twice for my September subscription. Please refund the extra 12,000.", "billing"),
  ("The app crashes every time I try to upload a photo on Android 14.",
  "technical"),
  ("I forgot my password and the reset email never arrives.",
  "account_access"),
  ("Do you have an office in Abuja? Just curious.",
  "other"),
  ("My card was declined but the money left my account and my plan is still inactive.", "billing"),
  ("Video calls freeze after about two minutes on WiFi.",
  "technical"),
  ("I want to delete my account and everything you have stored about me.",
  "account_access"),
  ("Please change the invoice address to my company name before the next receipt.",
  "billing"),
  ("Login says 'too many attempts' even though this is my first try today.",
  "account_access"),
  ("Your new logo looks great, just wanted to say well done.",
  "other"),
]

print(len(TEST_SET), "test cases loaded")

10 test cases loaded


In [ ]:
SYSTEM_V1 = "You are a helpful assistant."

def prompt_v1(text):
  return f"Classify this customer message: {text}"

for text, gold in TEST_SET[:3]:
  print("GOLD:", gold)
  print("RAW :", repr(ask(SYSTEM_V1, prompt_v1(text))))
  print("-" * 70)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GOLD: billing


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAW : "The customer message can be classified as an error report or complaint regarding billing issues related to their subscription service. The customer is reporting that they were charged twice for their September subscription and requesting a refund of the additional amount of $12,000. This type of message typically indicates a problem with the billing process or payment system, which may require further investigation by the company's support team to resolve."
----------------------------------------------------------------------
GOLD: technical


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAW : 'The given customer message can be classified as an issue or problem related to the functionality of an application (app) that is experiencing crashes when attempting to upload photos.\n\nSpecifically, it indicates:\n\n- **Application Name**: "The app"\n- **Functionality Issue**: Crashes ("crashes")\n- **Action Involved**: Attempting to upload a photo\n\nThis type of feedback typically suggests that there\'s a technical problem with the app\'s design or implementation that prevents it from functioning correctly under certain conditions. In this case, the user is encountering issues specifically related to uploading photos on their device running Android'
----------------------------------------------------------------------
GOLD: account_access
RAW : 'The customer message "I forgot my password and the reset email never arrives" can be classified as an issue with account recovery or security concerns related to forgotten passwords. This type of message typically indicates that the

**Observations:** The raw model outputs show several classic classification failures. The model sometimes invents labels such as “Refund Request” or “Customer Complaint” instead of using the predefined categories. It can also be overly conversational, providing explanations or reasoning when only a single label is required. The output format is inconsistent, with responses sometimes appearing as sentences, bullet points, or other formats. Because of these variations, the output is not reliably parseable by a downstream program. This shows the need for a stricter system prompt that defines the allowed labels and requires the model to return only one valid label in a consistent format.

In [ ]:
def parse_label(raw):
    """Pull a valid label out of the model's reply, or report failure."""
    match = re.search(r"\{.*\}", raw, re.S)
    # find the first {...} block
    if match:
        try:
            data = json.loads(match.group(0))
            label = str(data.get("label", "")).strip().lower()
            if label in LABELS:
                return label
        except json.JSONDecodeError:
            pass
    return "UNPARSEABLE"


def evaluate(system_prompt, build_user, sample=False):
    correct = parsed = 0

    for text, gold in TEST_SET:
        raw = ask(system_prompt, build_user(text), sample=sample)
        pred = parse_label(raw)
        parsed += (pred != "UNPARSEABLE")
        correct += (pred == gold)

        flag = "PASS" if pred == gold else "FAIL"
        print(f"{flag} gold={gold:<15} pred={pred:<15} {text[:45]}...")

    n = len(TEST_SET)
    print(f"\nParseable: {parsed}/{n}")
    return correct / n

In [ ]:
score_v1 = evaluate(SYSTEM_V1, prompt_v1)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FAIL gold=billing         pred=UNPARSEABLE     I was charged twice for my September subscrip...


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FAIL gold=technical       pred=UNPARSEABLE     The app crashes every time I try to upload a ...


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

FAIL gold=account_access  pred=UNPARSEABLE     I want to delete my account and everything yo...


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FAIL gold=billing         pred=UNPARSEABLE     Please change the invoice address to my compa...


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FAIL gold=account_access  pred=UNPARSEABLE     Login says 'too many attempts' even though th...
FAIL gold=other           pred=UNPARSEABLE     Your new logo looks great, just wanted to say...

Parseable: 0/10


In [ ]:
SYSTEM_V2 = """You are a support-ticket routing system for a mobile fintech app.
Classify each customer message into exactly one label.

Labels:
- billing: payments, charges, refunds, invoices, pricing, failed transactions
- technical: crashes, bugs, errors, freezing, anything broken in the product
- account_access: login, passwords, OTP codes, lockouts, creating or deleting an account
- other: anything else, including praise, general questions, and company information

Rules:
- If a message mentions both money and access, label it by what the customer wants FIXED.
- Never invent a label outside the four above.
- Reply with JSON only. No explanation, no markdown, no code fences.

Output format:
{"label": "billing | technical | account_access | other", "confidence": 0.0-1.0}"""

def prompt_v2(text):
  return f"Message: {text}"

score_v2 = evaluate(SYSTEM_V2, prompt_v2)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PASS gold=billing         pred=billing         I was charged twice for my September subscrip...


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PASS gold=technical       pred=technical       The app crashes every time I try to upload a ...


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

PASS gold=account_access  pred=account_access  I want to delete my account and everything yo...


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FAIL gold=billing         pred=account_access  Please change the invoice address to my compa...


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PASS gold=account_access  pred=account_access  Login says 'too many attempts' even though th...
PASS gold=other           pred=other           Your new logo looks great, just wanted to say...

Parseable: 10/10


In [ ]:
FEW_SHOT = """Examples:

Message: My subscription renewed but the app still shows the Free plan.
{"label": "billing", "confidence": 0.8}

Message: The camera screen is completely black on my Samsung.
{"label": "technical", "confidence": 0.95}

Message: The OTP code never arrives so I cannot sign in.
{"label": "account_access", "confidence": 0.95}

Message: Who founded this company?
{"label": "other", "confidence": 0.9}
"""

def prompt_v3(text):
  return f"{FEW_SHOT}\nNow classify this one.\n\nMessage: {text}"

score_v3 = evaluate(SYSTEM_V2, prompt_v3)
# same system prompt, richer user message

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PASS gold=billing         pred=billing         I was charged twice for my September subscrip...


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PASS gold=technical       pred=technical       The app crashes every time I try to upload a ...


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

PASS gold=account_access  pred=account_access  I want to delete my account and everything yo...


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PASS gold=billing         pred=billing         Please change the invoice address to my compa...


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PASS gold=account_access  pred=account_access  Login says 'too many attempts' even though th...
PASS gold=other           pred=other           Your new logo looks great, just wanted to say...

Parseable: 10/10


In [ ]:
def consistency(system_prompt, build_user, runs=3):
    stable = 0
    for text, gold in TEST_SET:
        preds = {
            parse_label(ask(system_prompt, build_user(text), sample=True))
            for _ in range(runs)}
        if len(preds) == 1:
            stable += 1
        else:
            print("UNSTABLE:", preds, "|", text[:50])
    print(f"\nStable on {stable}/{len(TEST_SET)} messages over {runs} runs")
    return stable / len(TEST_SET)


consistency(SYSTEM_V2, prompt_v3)

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more 


Stable on 10/10 messages over 3 runs


1.0

In [ ]:
def ask(system_prompt, user_message, sample=False, temperature=0.7, max_new_tokens=120):
    """Send one system prompt + one user message to the model, return the text reply."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",
         "content": user_message},
    ]
    kwargs = {"max_new_tokens": max_new_tokens, "return_full_text": False}
    if sample:
        kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        kwargs.update(do_sample=False)
    # greedy = same input, same output
    out = llm(messages, **kwargs)[0]["generated_text"]
    if isinstance(out, list):
        # some versions return the full chat
        out = out[-1]["content"]
    return out.strip()


# smoke test
print(ask("You are a terse assistant.", "Say OK and nothing else."))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OK
